In [1]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# saved chunks load karto (PDFs parat vachaychi garaj nahi)
with open("../rag/index/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
texts = [c["text"] for c in chunks]

# stop_words = "the, is, a" sarkhe shabd ignore, ngram (1,2) = ekek shabd ani don-don shabdanchya jodya
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
chunk_matrix = vectorizer.fit_transform(texts)   # prattek chunk cha vector
print(chunk_matrix.shape)                        # (chunks, unique shabd/jodya)

(190, 11000)


In [2]:
def search(question, k=3):
    q_vec = vectorizer.transform([question])                      # prashna cha vector
    scores = cosine_similarity(q_vec, chunk_matrix).ravel()       # prattek chunk shi similarity
    top = scores.argsort()[::-1][:k]                              # sarvat jast score che k chunks
    return [(round(float(scores[i]), 3), chunks[i]) for i in top]

def show(question, k=3):
    print("\nQ:", question)
    for score, c in search(question, k):
        print(f"  {score} | {c['source']} | p{c['page']} | {c['text'][:100]}...")

In [3]:
for q in [
    "What is eGFR?",
    "Which tests check for kidney disease?",
    "What blood pressure target is recommended for CKD?",
    "What is albuminuria?",
    "What is specific gravity?",
    "Who won the cricket world cup?",
]:
    show(q)


Q: What is eGFR?
  0.134 | CKD Stages G1-G5 (NKF).pdf | p4 | Stage G3b CKD Moderate to severe loss of kidney function (eGFR 30-44 for 3 months or more) Stage G4 ...
  0.128 | eGFR (NKF).pdf | p5 | eGFR of 60 -89 may mean early-stage kidney disease eGFR of 15 -59 may mean kidney disease eGFR below...
  0.106 | CKD Stages G1-G5 (NKF).pdf | p3 | If your eGFR stays below 60, or your uACR stays above 30 for 3 months or longer, it may be a sign of...

Q: Which tests check for kidney disease?
  0.285 | CDC Testing.pdf | p1 | nt is working. Serum creatinine Because your kidneys remove waste, toxins, and extra fluid from the ...
  0.178 | CDC Testing.pdf | p1 | w if your levels are normal. If you have abnormal albumin levels, your doctor may want to run furthe...
  0.141 | CDC Testing.pdf | p1 | tested. Urine tests One of the earliest signs of kidney disease is when protein leaks into your urin...

Q: What blood pressure target is recommended for CKD?
  0.197 | KDIGO 2024 Executive Summary.pdf